In [14]:
import scipy.io as sio
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
%pip install -q sysidentpy
import torch
import torch.nn as nn
from sysidentpy.neural_network import NARXNN
from sysidentpy.basis_function import Polynomial

data = sio.loadmat('/Users/raab/Documents/Project_ML_CV/Benchmark_EEG_small/Benchmark_EEG_small.mat')
u = data['data']['u'].item().flatten().reshape(-1, 1)
y = data['data']['y'].item().flatten().reshape(-1, 1)  # output: EEG corticale

fs = float(256)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Using device: cpu


In [15]:
split = int(len(u) * 0.8)
utrain, utest = u[:split], u[split:]
ytrain, ytest = y[:split], y[split:]

scaler_u = StandardScaler()
scaler_y = StandardScaler()

utrain_scaled = scaler_u.fit_transform(utrain)
ytrain_scaled = scaler_y.fit_transform(ytrain)

# Trasformiamo il test usando i parametri appresi dal train
utest_scaled = scaler_u.transform(utest)
ytest_scaled = scaler_y.transform(ytest)



In [16]:
class NARX_Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

In [17]:
ny = 5
nu = 5

# The input size to NARX_Net must match the number of regressors generated by sysidentpy.
net = NARX_Net(65)
model = NARXNN(
    net=net,
    ylag=ny,
    xlag=nu,
    basis_function=Polynomial(degree=2),
    loss_func='mse_loss',
    optimizer='Adam',
    optim_params={},
    epochs=300,
    learning_rate=0.001,
    batch_size=32,
)

In [18]:
print("Inizio addestramento...")
model.fit(X=utrain_scaled, y=ytrain_scaled)
print("Addestramento completato!")

Inizio addestramento...
Addestramento completato!


In [19]:
yhat_onestep_scaled = model.predict(X=utest_scaled, y=ytest_scaled, steps_ahead=1)

# Predizione Multi-Step (Simulazione): usa le proprie predizioni passate
yhat_multistep_scaled = model.predict(X=utest_scaled, y=ytest_scaled) # Default in sysidentpy

# 7. Riportiamo i dati alla scala originale per calcolare gli errori reali
yhat_onestep = scaler_y.inverse_transform(yhat_onestep_scaled)
yhat_multistep = scaler_y.inverse_transform(yhat_multistep_scaled)

In [20]:
# Use multi-step prediction for VAF and align lengths safely.
y_pred_vaf = yhat_multistep.flatten()
y_true_vaf = ytest.flatten()[-len(y_pred_vaf):]

vaf = (1 - np.var(y_true_vaf - y_pred_vaf) / np.var(y_true_vaf)) * 100
print(f"VAF: {vaf:.2f}%")

VAF: -14.76%


In [21]:
def calculate_nrmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse / (np.max(y_true) - np.min(y_true))

In [22]:
max_lag = max(ny, nu)
ytest_trimmed = ytest[max_lag:]

mse_1step = mean_squared_error(ytest_trimmed, yhat_onestep)
nrmse_1step = calculate_nrmse(ytest_trimmed, yhat_onestep)

mse_multi = mean_squared_error(ytest_trimmed, yhat_multistep)
nrmse_multi = calculate_nrmse(ytest_trimmed, yhat_multistep)

print(f"\n--- Risultati Predizione One-Step ---")
print(f"MSE: {mse_1step:.6f}")
print(f"NRMSE: {nrmse_1step:.4f} ({nrmse_1step*100:.2f}%)")

print(f"\n--- Risultati Predizione Multi-Step (Simulazione) ---")
print(f"MSE: {mse_multi:.6f}")
print(f"NRMSE: {nrmse_multi:.4f} ({nrmse_multi*100:.2f}%)")

ValueError: Found input variables with inconsistent numbers of samples: [3579, 3584]